In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import os
import sys
sys.path.insert(0, "/".join(["..", "python"]))

import warnings
import numpy as np
import rasterio
warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

from gridr.chain.grid_resampling_chain import basic_grid_resampling_chain
from gridr.misc.mandrill import mandrill
from notebook_utils import plot_im
from chain_notebook_utils import (
    write_array, create_grid, create_star_polygon, plot_grid_on_image,
)

IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
if not IN_DOC_BUILD:
    from bokeh.io import output_notebook
    output_notebook()

# File paths shared across the chain tutorial series.
RASTER_IN          = "./grid_resampling_chain_raster_in.tif"
GRID_IN_F64        = "./grid_resampling_chain_grid_in_f64.tif"
output_raster_path = "./grid_resampling_chain_output_raster.tif"
output_mask_path   = "./grid_resampling_chain_output_mask.tif"

# Grid parameters shared across the chain tutorial series.
nrow, ncol = 50, 40
origin_pos  = np.array((0.3, 0.2))
origin_node = np.array((0., 0.))
v_row_y, v_row_x = 5.2, 1.2
v_col_y, v_col_x = -2.7, 7.1

grid_row_f64, grid_col_f64 = create_grid(
    nrow, ncol, origin_pos, origin_node,
    v_row_y, v_row_x, v_col_y, v_col_x, dtype=np.float64,
)

# Hybrid input creation: ensure the source raster and grid file exist on disk.
if not os.path.exists(RASTER_IN):
    write_array(mandrill, dtype=mandrill.dtype, fileout=RASTER_IN)
if not os.path.exists(GRID_IN_F64):
    write_array(np.array([grid_row_f64, grid_col_f64]),
                dtype=np.float64, fileout=GRID_IN_F64)

# Default output-dataset open arguments (resolution-dependent overrides
# are applied per-notebook below).
grid_resolution = (8, 8)
from gridr.core.grid.grid_commons import grid_full_resolution_shape
output_shape = grid_full_resolution_shape(
    shape=grid_row_f64.shape, resolution=grid_resolution,
)
raster_out_open_args = {
    "driver": "GTiff", "dtype": np.float64,
    "height": output_shape[0], "width": output_shape[1], "count": 1,
}
mask_out_open_args = {
    "driver": "GTiff", "dtype": np.uint8,
    "height": output_shape[0], "width": output_shape[1],
    "count": 1, "nbits": 1,
}

grid_mask_in_path        = "./grid_resampling_chain_grid_mask_in_u8_1_0.tif"
grid_mask_in_path_i8     = "./grid_resampling_chain_grid_mask_in_i8_0_m10.tif"
raster_mask_in_path_u8   = "./grid_resampling_chain_raster_mask_in_u8_1_0.tif"

# Ensure the grid mask exists on disk (created in notebook 003 if missing).
grid_mask_in_valid_value, grid_mask_in_invalid_value = 1, 0
roi = np.array(((10, 40), (5, 100)))
grid_mask = np.full(grid_row_f64.shape, grid_mask_in_valid_value, dtype=np.uint8)
grid_mask[
    np.logical_and(
        np.logical_and(grid_row_f64 >= roi[0][0], grid_row_f64 <= roi[0][1]),
        np.logical_and(grid_col_f64 >= roi[1][0], grid_col_f64 <= roi[1][1]),
    )
] = grid_mask_in_invalid_value
if not os.path.exists(grid_mask_in_path):
    write_array(grid_mask, dtype=np.uint8, fileout=grid_mask_in_path)

# Ensure the source raster mask exists on disk (created in notebook 004 if missing).
array_src_mask_validity_valid, array_src_mask_validity_invalid = 1, 0
array_in_mask = np.full(mandrill[0].shape, array_src_mask_validity_valid, dtype=np.uint8)
array_in_mask[slice(50, 71), slice(150, 171)] = array_src_mask_validity_invalid
if not os.path.exists(raster_mask_in_path_u8):
    write_array(array_in_mask, dtype=np.uint8, fileout=raster_mask_in_path_u8)


# Grid shift and computation window

This notebook covers two parameters that change *which* grid samples are computed: `grid_shift` applies a global offset to all grid coordinate values, and `win` restricts the output to a sub-region of the full-resolution grid.

**What you'll learn**

- How `grid_shift` differs from `array_in_origin`
- How to apply a global coordinate offset to the grid
- How to restrict computation to a window with `win`
- How to size output datasets when using a window

## Setting things up

We reuse all previously prepared inputs: source raster, grid file, grid mask, source raster mask, and the geometry polygons from notebook 005.

In [ ]:
from shapely.geometry import Polygon

geometry_valid  = Polygon([(40, 10), (300, 30), (270, 200), (40, 200)])
invalid_polygon = create_star_polygon(100, 100, 50)

## Shifting the grid coordinates with `grid_shift`

`grid_shift` accepts a `(shift_row, shift_col)` tuple of integers or floats and applies it to all grid coordinates **before** any grid geometry computation.

This is different from `array_in_origin`, which is used internally by the chain to account for the current tile's origin within the source raster; `array_in_origin` does not affect grid geometry calculations, whereas `grid_shift` does.

In [ ]:
with rasterio.open(GRID_IN_F64, "r") as grid_in_ds, \
        rasterio.open(RASTER_IN, "r") as array_src_ds, \
        rasterio.open(grid_mask_in_path, "r") as grid_mask_in_ds, \
        rasterio.open(raster_mask_in_path_u8, "r") as raster_mask_in_ds, \
        rasterio.open(output_raster_path, "w", **raster_out_open_args) as array_out_ds, \
        rasterio.open(output_mask_path, "w", **mask_out_open_args) as mask_out_ds:

    basic_grid_resampling_chain(
        grid_ds                        = grid_in_ds,
        grid_row_coords_band           = 1,
        grid_col_coords_band           = 2,
        grid_resolution                = grid_resolution,
        array_src_ds                   = array_src_ds,
        array_src_bands                = 1,
        array_out_ds                   = array_out_ds,
        interp                         = "cubic",
        nodata_out                     = 0,
        win                            = None,
        grid_shift                     = (50.5, 30.),    # <= global coordinate offset
        mask_out_ds                    = mask_out_ds,
        grid_mask_in_ds                = grid_mask_in_ds,
        grid_mask_in_unmasked_value    = grid_mask_in_valid_value,
        grid_mask_in_band              = 1,
        array_src_mask_ds              = raster_mask_in_ds,
        array_src_mask_band            = 1,
        array_src_mask_validity_pair   = (array_src_mask_validity_valid,
                                          array_src_mask_validity_invalid),
        array_src_geometry_origin      = (0., 0.),
        array_src_geometry_pair        = [geometry_valid, invalid_polygon],
    )

In [ ]:
with rasterio.open(output_raster_path, "r") as raster_ds, \
        rasterio.open(output_mask_path, "r") as mask_ds:
    plot_im({"output image": raster_ds.read(1), "output mask": mask_ds.read(1)},
            prefix="grid_shift_output")

The shift is applied uniformly to all grid coordinates; the grid mask pattern itself is unaffected.

## Limiting computation with `win`

The `win` parameter restricts the output to a sub-region of the full-resolution grid. Indices are expressed in the full-resolution output coordinate frame, **after** `grid_resolution` oversampling. The output datasets must be sized to match the window.

In [ ]:
win = np.asarray([[120, 280], [70, 240]])

In [ ]:
plot_grid_on_image(
    1.5, grid_row_f64, grid_col_f64, (10, 10),
    (mandrill.shape[1], mandrill.shape[2]),
    mask=grid_mask, win=win, raster_image=mandrill[0],
    raster_image_mask=array_in_mask,
    geometry_origin=(0., 0.),
    geometry_pair=[geometry_valid, invalid_polygon],
    prefix="window_input",
)

The orange dashed rectangle marks the requested computation window. The output dataset sizes must be adjusted accordingly.

In [ ]:
raster_out_open_args_win = {
    "driver": "GTiff", "dtype": np.float64,
    "height": win[0][1] - win[0][0] + 1,
    "width":  win[1][1] - win[1][0] + 1,
    "count":  1,
}
mask_out_open_args_win = {
    "driver": "GTiff", "dtype": np.uint8,
    "height": win[0][1] - win[0][0] + 1,
    "width":  win[1][1] - win[1][0] + 1,
    "count":  1, "nbits": 1,
}

with rasterio.open(GRID_IN_F64, "r") as grid_in_ds, \
        rasterio.open(RASTER_IN, "r") as array_src_ds, \
        rasterio.open(grid_mask_in_path, "r") as grid_mask_in_ds, \
        rasterio.open(raster_mask_in_path_u8, "r") as raster_mask_in_ds, \
        rasterio.open(output_raster_path, "w", **raster_out_open_args_win) as array_out_ds, \
        rasterio.open(output_mask_path, "w", **mask_out_open_args_win) as mask_out_ds:

    basic_grid_resampling_chain(
        grid_ds                        = grid_in_ds,
        grid_row_coords_band           = 1,
        grid_col_coords_band           = 2,
        grid_resolution                = grid_resolution,
        array_src_ds                   = array_src_ds,
        array_src_bands                = 1,
        array_out_ds                   = array_out_ds,
        interp                         = "cubic",
        nodata_out                     = 0,
        win                            = win,            # <= restrict to a sub-window
        mask_out_ds                    = mask_out_ds,
        grid_mask_in_ds                = grid_mask_in_ds,
        grid_mask_in_unmasked_value    = grid_mask_in_valid_value,
        grid_mask_in_band              = 1,
        array_src_mask_ds              = raster_mask_in_ds,
        array_src_mask_band            = 1,
        array_src_mask_validity_pair   = (array_src_mask_validity_valid,
                                          array_src_mask_validity_invalid),
        array_src_geometry_origin      = (0., 0.),
        array_src_geometry_pair        = [geometry_valid, invalid_polygon],
    )

In [ ]:
with rasterio.open(output_raster_path, "r") as raster_ds, \
        rasterio.open(output_mask_path, "r") as mask_ds:
    plot_im({"output image": raster_ds.read(1), "output mask": mask_ds.read(1)},
            prefix="apply_window_output")